## SRP057462

**paper:** [PMID: 26485701](https://pmc.ncbi.nlm.nih.gov/articles/PMC4618353/) - Sex Bias and Maternal Contribution to Gene Expression Divergence in Drosophila Blastoderm Embryos, 2015

**date, curator:** 2026-06-26, Sara Carsanaro

**notes**
* 2 samples removed from analysis (see supplemental table), so i also rejected those samples
    * GSM1662157 and GSM1662158
* blastoderm stage aka stage 5
* there is an unfertilized egg stage for fly dev ontology but not in uberon, annotating as life cycle for uberon

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,unfertilized egg,CL:0000025,egg cell,perfect match
5,unfertilized egg,FBbt:00015288,unfertilized egg,perfect match
7,Whole embryo,UBERON:0000307,blastula,perfect match
23,Whole embryo,FBbt:00005304,blastoderm embryo,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,unfertilized egg,UBERON:0000104,life cycle,other
5,unfertilized egg,FBdv:00005287,unfertilized egg stage,perfect match
7,Blastoderm,UBERON:0000108,blastula stage,perfect match
23,Blastoderm,FBdv:00005304,blastoderm stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP057462"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [7]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1000515,SRP057462,Illumina HiSeq 2000,SRS914451,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662170,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,,,,,,vir.U.2,"SAMN03492554,GSM1662170",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. virilis_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
1,SRX1000514,SRP057462,Illumina HiSeq 2000,SRS914450,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662169,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,,,,,,vir.U.1,"SAMN03492556,GSM1662169",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. virilis_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
2,SRX1000513,SRP057462,Illumina HiSeq 2000,SRS914454,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662168,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,MV2-25 / Tucson 14011-0121.94,,7237,,,,,,pse.U.3,"SAMN03492555,GSM1662168",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. pseudoobscura_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
3,SRX1000512,SRP057462,Illumina HiSeq 2000,SRS914452,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662167,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,,,,,,yak.U.2,"SAMN03492553,GSM1662167",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. yakuba_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
4,SRX1000511,SRP057462,Illumina HiSeq 2000,SRS914453,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662166,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,,,,,,yak.U.1,"SAMN03492552,GSM1662166",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. yakuba_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
5,SRX1000510,SRP057462,Illumina HiSeq 2000,SRS914455,FBbt:00015288,unfertilized egg,FBdv:00005287,unfertilized egg stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662165,unfertilized egg,unfertilized egg,perfect match,full sampling,perfect match,NA,Oregon-R,,7227,,,,,,mel.U.3,"SAMN03492551,GSM1662165",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. melanogaster_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
6,SRX1000509,SRP057462,Illumina HiSeq 2000,SRS914456,FBbt:00015288,unfertilized egg,FBdv:00005287,unfertilized egg stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662164,unfertilized egg,unfertilized egg,perfect match,full sampling,perfect match,NA,Oregon-R,,7227,,,,,,mel.U.2,"SAMN03492550,GSM1662164",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen)

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [8]:
unique_sorted(library, "infoOrgan")

['Whole embryo' 'unfertilized egg']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [9]:
unique_sorted(library, "infoStage")

['Blastoderm' 'unfertilized egg']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1000515,SRP057462,Illumina HiSeq 2000,SRS914451,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662170,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.2,"SAMN03492554,GSM1662170",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. virilis_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
1,SRX1000514,SRP057462,Illumina HiSeq 2000,SRS914450,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662169,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.1,"SAMN03492556,GSM1662169",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. virilis_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
2,SRX1000513,SRP057462,Illumina HiSeq 2000,SRS914454,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662168,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,MV2-25 / Tucson 14011-0121.94,,7237,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,pse.U.3,"SAMN03492555,GSM1662168",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. pseudoobscura_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
3,SRX1000512,SRP057462,Illumina HiSeq 2000,SRS914452,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662167,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.2,"SAMN03492553,GSM1662167",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. yakuba_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
4,SRX1000511,SRP057462,Illumina HiSeq 2000,SRS914453,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662166,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.1,"SAMN03492552,GSM1662166",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. yakuba_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
5,SRX1000510,SRP057462,Illumina HiSeq 2000,SRS914455,FBbt:00015288,unfertilized egg,FBdv:00005287,unfertilized egg stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662165,unfertilized egg,unfertilized egg,perfect match,full sampling,perfect match,NA,Oregon-R,,7227,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,mel.U.3,"SAMN03492551,GSM1662165",,,,,,,,,,,26/06/2026,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. melanogaster_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
6,SRX1000509,SRP057462,Illumina HiSeq 2000,SRS914456,FBbt:00015288,unfert

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-26'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1000515,SRP057462,Illumina HiSeq 2000,SRS914451,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662170,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.2,"SAMN03492554,GSM1662170",,,,,,,,,,SAC,2026-06-26,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. virilis_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
1,SRX1000514,SRP057462,Illumina HiSeq 2000,SRS914450,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662169,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.1,"SAMN03492556,GSM1662169",,,,,,,,,,SAC,2026-06-26,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. virilis_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
2,SRX1000513,SRP057462,Illumina HiSeq 2000,SRS914454,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662168,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,MV2-25 / Tucson 14011-0121.94,,7237,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,pse.U.3,"SAMN03492555,GSM1662168",,,,,,,,,,SAC,2026-06-26,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. pseudoobscura_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
3,SRX1000512,SRP057462,Illumina HiSeq 2000,SRS914452,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662167,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.2,"SAMN03492553,GSM1662167",,,,,,,,,,SAC,2026-06-26,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. yakuba_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
4,SRX1000511,SRP057462,Illumina HiSeq 2000,SRS914453,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662166,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.1,"SAMN03492552,GSM1662166",,,,,,,,,,SAC,2026-06-26,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. yakuba_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
5,SRX1000510,SRP057462,Illumina HiSeq 2000,SRS914455,FBbt:00015288,unfertilized egg,FBdv:00005287,unfertilized egg stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662165,unfertilized egg,unfertilized egg,perfect match,full sampling,perfect match,NA,Oregon-R,,7227,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,mel.U.3,"SAMN03492551,GSM1662165",,,,,,,,,,SAC,2026-06-26,"Trizol extraction (Invitrogen), according to the manufacturer's instructions. Liraries were made using the standard Illumina Truseq protocol",,,,D. melanogaster_Whole embryo,,,,TRANSCRIPTOMIC,cDNA
6,SRX1000509,SRP057462,Illumina HiSeq 2000,SRS914456,FB

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 26485701'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX1000515,SRP057462,Illumina HiSeq 2000,SRS914451,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662170,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.2,"SAMN03492554,GSM1662170",,,,,,,PMID: 26485701,,,SAC,2026-06-26
1,SRX1000514,SRP057462,Illumina HiSeq 2000,SRS914450,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662169,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.1,"SAMN03492556,GSM1662169",,,,,,,PMID: 26485701,,,SAC,2026-06-26
2,SRX1000513,SRP057462,Illumina HiSeq 2000,SRS914454,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662168,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,MV2-25 / Tucson 14011-0121.94,,7237,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,pse.U.3,"SAMN03492555,GSM1662168",,,,,,,PMID: 26485701,,,SAC,2026-06-26
3,SRX1000512,SRP057462,Illumina HiSeq 2000,SRS914452,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662167,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.2,"SAMN03492553,GSM1662167",,,,,,,PMID: 26485701,,,SAC,2026-06-26
4,SRX1000511,SRP057462,Illumina HiSeq 2000,SRS914453,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662166,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.1,"SAMN03492552,GSM1662166",,,,,,,PMID: 26485701,,,SAC,2026-06-26
5,SRX1000510,SRP057462,Illumina HiSeq 2000,SRS914455,FBbt:00015288,unfertilized egg,FBdv:00005287,unfertilized egg stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662165,unfertilized egg,unfertilized egg,perfect match,full sampling,perfect match,NA,Oregon-R,,7227,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,mel.U.3,"SAMN03492551,GSM1662165",,,,,,,PMID: 26485701,,,SAC,2026-06-26
6,SRX1000509,SRP057462,Illumina HiSeq 2000,SRS914456,FBbt:00015288,unfertilized egg,FBdv:00005287,unfertilized egg stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662164,unfertilized egg,unfertilized egg,perfect match,full sampling,perfect match,NA,Oregon-R,,7227,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,mel.U.2,"SAMN03492550,GSM1662164",,,,,,,PMID: 26485701,,,SAC,2026-06-26
7,SRX1000508,SRP057462,Illumina HiSeq 2000,SRS914457,UBERON:0000307,blastula,UBERON:0000108,blastula stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662163,Whole embryo,Blastoderm,perfect match,full sampling,perfect match,F,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.F.3,"SAMN03492549,GSM1662163",,,,,,,PMID: 26485701,,,SAC,2026-06-26
8,SRX1000507,SRP057462,Illumina HiSeq 2000,SRS914458,UBERON:0000307,blastula,UBERON:0000108,blastula stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1662162,Whole embryo,Blastoderm,perfect match,full sampling,perfect match,F,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.F.2,"SAMN03492548,GSM1662162",,,,,,,PMID: 26485701,,,SAC,2026-06-26
9,SRX1000506,SRP057462,Illumina HiSeq 2000,SRS914459,UBERON:0000307,blastula,UBERON:0000108,blastula stage,http://www.ncbi.nlm.nih.gov/geo/

### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP057462,Sex bias and maternal contribution to gene expression divergence in Drosophila blastoderm embryos,"Early embryogenesis is a unique developmental stage where genetic control of development is handed off from mother to zygote. Yet the contribution of this transition to the evolution of gene expression is poorly understood. Here we study two aspects of gene expression specific to early embryogenesis in Drosophila: sex-biased gene expression prior to the onset of canonical X chromosomal dosage compensation, and the contribution of maternally supplied mRNAs. We sequenced mRNAs from individual unfertilized eggs, precisely staged and sexed blastoderm embryos and compared levels between D. melanogaster, D. yakuba, D. pseudoobscura and D. virilis. First, we find that mRNA content is highly conserved for a given stage and that studies relying on pooled embryos may systematically overstate the degree of gene expression divergence. Unlike studies done on larvae and adults, where most species show a larger proportion of genes with male-biased expression, we find that transcripts in Drosophila embryos are largely female-biased in all species, likely due to incomplete dosage compensation prior to the activation of the canonical dosage compensation mechanism. The divergence of sex-biased gene expression across species is observed to be often due to a decrease of expression, the most drastic example of which is the overall reduction of male expression from the neo-X chromosome in D. pseudoobscura, leading to a pervasive female-bias on this chromosome. We see no evidence for a faster evolution of expression on the X chromosome in embryos (no “faster X” effect), unlike in adults and contrary to a previous study on pooled non-sexed embryos. Finally, we find that most genes are conserved in regard to their maternal or zygotic origin of transcription, and present evidence that differences in maternal contribution to the blastoderm transcript pool may be due to species-specific divergence of transcript degradation rates Overall design: We sequenced mRNA from D. melanogaser, D. yakuba, D. pseudoobscura and D. virilis single unfertilized eggs (1 to 2 per species) and from both single female and male embryos (3 per sex per species).",SRA,,,,,,GSE68062,PRJNA281640,26485701,,10.1371/journal.pgen.1005592,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

29

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP057462,Sex bias and maternal contribution to gene expression divergence in Drosophila blastoderm embryos,"Early embryogenesis is a unique developmental stage where genetic control of development is handed off from mother to zygote. Yet the contribution of this transition to the evolution of gene expression is poorly understood. Here we study two aspects of gene expression specific to early embryogenesis in Drosophila: sex-biased gene expression prior to the onset of canonical X chromosomal dosage compensation, and the contribution of maternally supplied mRNAs. We sequenced mRNAs from individual unfertilized eggs, precisely staged and sexed blastoderm embryos and compared levels between D. melanogaster, D. yakuba, D. pseudoobscura and D. virilis. First, we find that mRNA content is highly conserved for a given stage and that studies relying on pooled embryos may systematically overstate the degree of gene expression divergence. Unlike studies done on larvae and adults, where most species show a larger proportion of genes with male-biased expression, we find that transcripts in Drosophila embryos are largely female-biased in all species, likely due to incomplete dosage compensation prior to the activation of the canonical dosage compensation mechanism. The divergence of sex-biased gene expression across species is observed to be often due to a decrease of expression, the most drastic example of which is the overall reduction of male expression from the neo-X chromosome in D. pseudoobscura, leading to a pervasive female-bias on this chromosome. We see no evidence for a faster evolution of expression on the X chromosome in embryos (no “faster X” effect), unlike in adults and contrary to a previous study on pooled non-sexed embryos. Finally, we find that most genes are conserved in regard to their maternal or zygotic origin of transcription, and present evidence that differences in maternal contribution to the blastoderm transcript pool may be due to species-specific divergence of transcript degradation rates Overall design: We sequenced mRNA from D. melanogaser, D. yakuba, D. pseudoobscura and D. virilis single unfertilized eggs (1 to 2 per species) and from both single female and male embryos (3 per sex per species).",SRA,partial,Bgee 1K,29,TruSeq RNA Sample Preparation Kit,full_length,GSE68062,PRJNA281640,26485701,,10.1371/journal.pgen.1005592,,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '26485701'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4618353/'
experiment.loc[:,'DOI'] = '10.1371/journal.pgen.1005592'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP057462,Sex bias and maternal contribution to gene expression divergence in Drosophila blastoderm embryos,"Early embryogenesis is a unique developmental stage where genetic control of development is handed off from mother to zygote. Yet the contribution of this transition to the evolution of gene expression is poorly understood. Here we study two aspects of gene expression specific to early embryogenesis in Drosophila: sex-biased gene expression prior to the onset of canonical X chromosomal dosage compensation, and the contribution of maternally supplied mRNAs. We sequenced mRNAs from individual unfertilized eggs, precisely staged and sexed blastoderm embryos and compared levels between D. melanogaster, D. yakuba, D. pseudoobscura and D. virilis. First, we find that mRNA content is highly conserved for a given stage and that studies relying on pooled embryos may systematically overstate the degree of gene expression divergence. Unlike studies done on larvae and adults, where most species show a larger proportion of genes with male-biased expression, we find that transcripts in Drosophila embryos are largely female-biased in all species, likely due to incomplete dosage compensation prior to the activation of the canonical dosage compensation mechanism. The divergence of sex-biased gene expression across species is observed to be often due to a decrease of expression, the most drastic example of which is the overall reduction of male expression from the neo-X chromosome in D. pseudoobscura, leading to a pervasive female-bias on this chromosome. We see no evidence for a faster evolution of expression on the X chromosome in embryos (no “faster X” effect), unlike in adults and contrary to a previous study on pooled non-sexed embryos. Finally, we find that most genes are conserved in regard to their maternal or zygotic origin of transcription, and present evidence that differences in maternal contribution to the blastoderm transcript pool may be due to species-specific divergence of transcript degradation rates Overall design: We sequenced mRNA from D. melanogaser, D. yakuba, D. pseudoobscura and D. virilis single unfertilized eggs (1 to 2 per species) and from both single female and male embryos (3 per sex per species).",SRA,partial,Bgee 1K,29,TruSeq RNA Sample Preparation Kit,full_length,GSE68062,PRJNA281640,26485701,https://pmc.ncbi.nlm.nih.gov/articles/PMC4618353/,10.1371/journal.pgen.1005592,,


#### comments

In [19]:
experiment.loc[:,'comment'] = 'rejected 2 samples, low quality per methods'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP057462,Sex bias and maternal contribution to gene expression divergence in Drosophila blastoderm embryos,"Early embryogenesis is a unique developmental stage where genetic control of development is handed off from mother to zygote. Yet the contribution of this transition to the evolution of gene expression is poorly understood. Here we study two aspects of gene expression specific to early embryogenesis in Drosophila: sex-biased gene expression prior to the onset of canonical X chromosomal dosage compensation, and the contribution of maternally supplied mRNAs. We sequenced mRNAs from individual unfertilized eggs, precisely staged and sexed blastoderm embryos and compared levels between D. melanogaster, D. yakuba, D. pseudoobscura and D. virilis. First, we find that mRNA content is highly conserved for a given stage and that studies relying on pooled embryos may systematically overstate the degree of gene expression divergence. Unlike studies done on larvae and adults, where most species show a larger proportion of genes with male-biased expression, we find that transcripts in Drosophila embryos are largely female-biased in all species, likely due to incomplete dosage compensation prior to the activation of the canonical dosage compensation mechanism. The divergence of sex-biased gene expression across species is observed to be often due to a decrease of expression, the most drastic example of which is the overall reduction of male expression from the neo-X chromosome in D. pseudoobscura, leading to a pervasive female-bias on this chromosome. We see no evidence for a faster evolution of expression on the X chromosome in embryos (no “faster X” effect), unlike in adults and contrary to a previous study on pooled non-sexed embryos. Finally, we find that most genes are conserved in regard to their maternal or zygotic origin of transcription, and present evidence that differences in maternal contribution to the blastoderm transcript pool may be due to species-specific divergence of transcript degradation rates Overall design: We sequenced mRNA from D. melanogaser, D. yakuba, D. pseudoobscura and D. virilis single unfertilized eggs (1 to 2 per species) and from both single female and male embryos (3 per sex per species).",SRA,partial,Bgee 1K,29,TruSeq RNA Sample Preparation Kit,full_length,GSE68062,PRJNA281640,26485701,https://pmc.ncbi.nlm.nih.gov/articles/PMC4618353/,10.1371/journal.pgen.1005592,,"rejected 2 samples, low quality per methods"


#### save complete file

In [20]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [21]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [22]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [25]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [26]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
66751,SRX3910045,SRP139084,Illumina HiSeq 2000,SRS3145701,UBERON:0000307,blastula,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,single embryo,stage 5,perfect match,full sampling,perfect match,NA,,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,D. yakuba_stage5_replicate2,SAMN08899807,,,,,,,PMID: 30557299,,,SAC,2026-06-26
66752,SRX3910046,SRP139084,Illumina HiSeq 2000,SRS3145702,UBERON:0000307,blastula,UBERON:0000108,blastula stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,single embryo,stage 5,perfect match,full sampling,perfect match,NA,,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,D. yakuba_stage5_replicate3,SAMN08899806,,,,,,,PMID: 30557299,,,SAC,2026-06-26
66753,SRX1000515,SRP057462,Illumina HiSeq 2000,SRS914451,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.2,"SAMN03492554,GSM1662170",,,,,,,PMID: 26485701,,,SAC,2026-06-26
66754,SRX1000514,SRP057462,Illumina HiSeq 2000,SRS914450,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,,,7244,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,vir.U.1,"SAMN03492556,GSM1662169",,,,,,,PMID: 26485701,,,SAC,2026-06-26
66755,SRX1000513,SRP057462,Illumina HiSeq 2000,SRS914454,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,MV2-25 / Tucson 14011-0121.94,,7237,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,pse.U.3,"SAMN03492555,GSM1662168",,,,,,,PMID: 26485701,,,SAC,2026-06-26
66756,SRX1000512,SRP057462,Illumina HiSeq 2000,SRS914452,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.2,"SAMN03492553,GSM1662167",,,,,,,PMID: 26485701,,,SAC,2026-06-26
66757,SRX1000511,SRP057462,Illumina HiSeq 2000,SRS914453,CL:0000025,egg cell,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,unfertilized egg,unfertilized egg,perfect match,full sampling,other,NA,Tai18E2 / Tucson 14021-0261.01,,7245,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,yak.U.1,"SAMN03492552,GSM1662166",,,,,,,PMID: 26485701,,,SAC,2026-06-26


In [27]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1271,SRP108530,RNA-seq of sexed adult tissues/body parts from...,RNA-seq of sexed adult tissues/body parts from...,SRA,partial,Bgee 1K,520,TruSeq Stranded mRNA,full_length,GSE99574,PRJNA388952,30599046,https://pmc.ncbi.nlm.nih.gov/articles/PMC6305970/,10.26508/lsa.201800156,"31558581[PMID],33563972[PMID]",removed ERCC spike ins and mixed species samples
1272,SRP139084,Evolution of maternal and zygotic mRNA complem...,The earliest stage of animal development is co...,SRA,partial,Bgee 1K,95,TruSeq RNA Sample Preparation Kit,full_length,GSE112858,PRJNA449374,30557299,https://pmc.ncbi.nlm.nih.gov/articles/PMC6312346/,10.1371/journal.pgen.1007838,32226006[PMID],"removed reanalysis samples from SRP057462, wil..."
1273,SRP057462,Sex bias and maternal contribution to gene exp...,Early embryogenesis is a unique developmental ...,SRA,partial,Bgee 1K,29,TruSeq RNA Sample Preparation Kit,full_length,GSE68062,PRJNA281640,26485701,https://pmc.ncbi.nlm.nih.gov/articles/PMC4618353/,10.1371/journal.pgen.1005592,,"rejected 2 samples, low quality per methods"


### add annotations to git

In [28]:
! git pull

Already up to date.


In [29]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop bd8ee22] adding annotated bulk experiment SRP057462
 2 files changed, 30 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.91 KiB | 1000.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   7b9f394..bd8ee22  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push